# Metro Rail Passenger Satisfaction: End-to-End ML Project

**Interview portfolio project based on the attached Metro Rail business case.**

A directly downloadable public Metro survey with the exact presentation schema was not verified. This notebook therefore uses the public Kaggle **Airline Passenger Satisfaction** dataset as a transparent proxy. The dataset has aligned demographics, travel purpose, service ratings, delay fields, and a satisfaction target. Airline field names are retained so the data is not misrepresented as actual Metro observations.

## Objectives
1. Predict passenger satisfaction.
2. Identify dissatisfaction drivers.
3. Compare Decision Tree, Random Forest, and XGBoost.
4. Produce explainable insights and a reusable model artifact.

**Public source:** Kaggle dataset `teejmahal20/airline-passenger-satisfaction`.

In [ ]:
!pip -q install kagglehub xgboost joblib

In [ ]:
import re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
 classification_report, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import joblib
warnings.filterwarnings('ignore'); RANDOM_STATE=42
pd.set_option('display.max_columns', 100)

## 1. Acquire public data
The loader uses KaggleHub and falls back to a Colab upload prompt if access is restricted.

In [ ]:
def load_public_data():
    try:
        import kagglehub
        root=Path(kagglehub.dataset_download('teejmahal20/airline-passenger-satisfaction'))
        files=list(root.rglob('*.csv'))
        if not files: raise FileNotFoundError('No CSV found')
        train=[p for p in files if 'train' in p.name.lower()]
        test=[p for p in files if 'test' in p.name.lower()]
        selected=(train[:1] or files[:1])+test[:1]
        df=pd.concat([pd.read_csv(p) for p in selected], ignore_index=True, sort=False)
        print(f'Loaded {len(df):,} rows from {root}')
        return df
    except Exception as e:
        print('Automatic download failed:', e)
        from google.colab import files
        uploaded=files.upload()
        names=[n for n in uploaded if n.lower().endswith('.csv')]
        if not names: raise ValueError('Upload train.csv and optionally test.csv')
        return pd.concat([pd.read_csv(n) for n in names], ignore_index=True, sort=False)
raw_df=load_public_data(); raw_df.head()

## 2. Audit, clean, and encode the target

In [ ]:
def clean_name(c): return re.sub(r'[^a-z0-9]+','_',str(c).strip().lower()).strip('_')
df=raw_df.copy(); df.columns=[clean_name(c) for c in df.columns]
df=df.loc[:,~df.columns.str.startswith('unnamed')].drop_duplicates().reset_index(drop=True)
target=next((c for c in ['satisfaction','satisfaction_v2'] if c in df.columns),None)
if target is None: raise KeyError(f'Satisfaction target not found: {df.columns.tolist()}')
labels=df[target].astype(str).str.strip().str.lower()
y=labels.isin({'satisfied','satisfaction','1','yes','true'}).astype(int)
X=df.drop(columns=[target,'id'],errors='ignore')
print('Shape:',df.shape); display(df.isna().sum().sort_values(ascending=False).head(10).to_frame('missing'))
display(pd.DataFrame({'label':labels,'encoded':y}).drop_duplicates()); display(y.value_counts().to_frame('count'))

## 3. Exploratory analysis

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4))
sns.countplot(x=y.map({0:'Neutral/Dissatisfied',1:'Satisfied'}),ax=ax[0]); ax[0].set_title('Satisfaction Distribution'); ax[0].tick_params(axis='x',rotation=15)
if {'departure_delay_in_minutes','arrival_delay_in_minutes'}.issubset(X.columns):
 s=X.sample(min(5000,len(X)),random_state=RANDOM_STATE); s=s.assign(satisfaction=y.loc[s.index].map({0:'Neutral/Dissatisfied',1:'Satisfied'}))
 sns.scatterplot(data=s,x='departure_delay_in_minutes',y='arrival_delay_in_minutes',hue='satisfaction',alpha=.4,ax=ax[1]); ax[1].set_title('Arrival vs Departure Delay')
plt.tight_layout(); plt.show()
for c in [x for x in ['gender','customer_type','type_of_travel','class'] if x in X.columns]:
 t=pd.DataFrame({c:X[c],'satisfied':y}).groupby(c,dropna=False).satisfied.agg(['mean','count']).sort_values('count',ascending=False); t['mean']=(100*t['mean']).round(2); display(t.rename(columns={'mean':'satisfaction_rate_pct'}))

## 4. Leakage-safe preprocessing and split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
num=X_train.select_dtypes(include=np.number).columns.tolist(); cat=X_train.select_dtypes(exclude=np.number).columns.tolist()
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat)])
print('Train/Test:',X_train.shape,X_test.shape,'Numeric/Categorical:',len(num),len(cat))

## 5. Train and compare models

In [ ]:
models={'Decision Tree':DecisionTreeClassifier(max_depth=12,min_samples_leaf=5,class_weight='balanced',random_state=RANDOM_STATE),'Random Forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced_subsample',n_jobs=-1,random_state=RANDOM_STATE),'XGBoost':XGBClassifier(n_estimators=400,max_depth=6,learning_rate=.08,subsample=.85,colsample_bytree=.85,objective='binary:logistic',eval_metric='logloss',n_jobs=-1,random_state=RANDOM_STATE)}
fitted={}; rows=[]
for name,m in models.items():
 p=Pipeline([('preprocess',clone(pre)),('model',m)]).fit(X_train,y_train); pred=p.predict(X_test); prob=p.predict_proba(X_test)[:,1]; fitted[name]=p
 rows.append({'model':name,'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),'roc_auc':roc_auc_score(y_test,prob)})
results=pd.DataFrame(rows).sort_values(['f1','roc_auc'],ascending=False).reset_index(drop=True); display(results.style.format({c:'{:.4f}' for c in results if c!='model'}))
best_name=results.loc[0,'model']; best=fitted[best_name]; print('Selected:',best_name)

## 6. Cross-validation and detailed evaluation

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)
s=cross_validate(best,X,y,cv=cv,scoring=['accuracy','precision','recall','f1','roc_auc'],n_jobs=-1)
cv_summary=pd.DataFrame({k.replace('test_',''):[v.mean(),v.std()] for k,v in s.items() if k.startswith('test_')},index=['mean','std']).T; display(cv_summary.style.format('{:.4f}'))
pred=best.predict(X_test); prob=best.predict_proba(X_test)[:,1]; print(classification_report(y_test,pred,target_names=['Neutral/Dissatisfied','Satisfied']))
fig,ax=plt.subplots(1,3,figsize=(17,4)); sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cmap='Blues',ax=ax[0]); ax[0].set_title(f'{best_name}: Confusion Matrix'); RocCurveDisplay.from_predictions(y_test,prob,ax=ax[1]); PrecisionRecallDisplay.from_predictions(y_test,prob,ax=ax[2]); plt.tight_layout(); plt.show()

## 7. Explainability and error analysis

In [ ]:
n=min(5000,len(X_test)); Xi=X_test.sample(n,random_state=RANDOM_STATE); yi=y_test.loc[Xi.index]
pi=permutation_importance(best,Xi,yi,scoring='f1',n_repeats=5,random_state=RANDOM_STATE,n_jobs=-1)
importance=pd.DataFrame({'feature':Xi.columns,'importance_mean':pi.importances_mean,'importance_std':pi.importances_std}).sort_values('importance_mean',ascending=False); display(importance.head(15))
top=importance.head(15).sort_values('importance_mean'); plt.figure(figsize=(9,6)); plt.barh(top.feature,top.importance_mean,xerr=top.importance_std); plt.title('Permutation Importance'); plt.tight_layout(); plt.show()
errors=X_test.copy(); errors['actual']=y_test; errors['predicted']=pred; errors['probability']=prob; display(errors[errors.actual!=errors.predicted].head(10))

## 8. Threshold tuning and artifact export

In [ ]:
thresholds=np.arange(.10,.91,.01)
table=pd.DataFrame([{'threshold':t,'precision':precision_score(y_test,prob>=t,zero_division=0),'recall':recall_score(y_test,prob>=t,zero_division=0),'f1':f1_score(y_test,prob>=t,zero_division=0)} for t in thresholds])
best_threshold=float(table.loc[table.f1.idxmax(),'threshold']); display(table.sort_values('f1',ascending=False).head())
artifact={'pipeline':best,'threshold':best_threshold,'target_mapping':{0:'neutral_or_dissatisfied',1:'satisfied'},'input_columns':X.columns.tolist(),'model_name':best_name}
joblib.dump(artifact,'metro_passenger_satisfaction_model.joblib'); results.to_csv('model_comparison.csv',index=False); importance.to_csv('feature_importance.csv',index=False)
print('Artifacts saved.')

def predict_passenger(record,artifact_path='metro_passenger_satisfaction_model.joblib'):
 a=joblib.load(artifact_path); row=pd.DataFrame([record]).reindex(columns=a['input_columns']); p=float(a['pipeline'].predict_proba(row)[:,1][0]); label=int(p>=a['threshold']); return {'prediction':a['target_mapping'][label],'satisfaction_probability':round(p,4),'threshold':a['threshold']}

## 9. Interview interpretation and production roadmap

Use measured notebook outputs rather than hard-coded results. Discuss the top importance drivers, low-satisfaction segments, model trade-offs, and error types. The public airline dataset is a workflow proxy and must be replaced by representative Metro survey data before operational use.

Recommended production extensions include data-contract validation, experiment tracking, hyperparameter search, fairness and drift monitoring, API or batch inference, automated tests, CI/CD, and governed retraining. These are roadmap items, not capabilities claimed as already implemented.